In [89]:
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

import html
import time
from bs4 import BeautifulSoup
import re
import pandas as pd
import os
from dotenv import load_dotenv
import time
from IPython.display import clear_output
import traceback

import os
import json
import requests
from urllib.parse import urljoin

BASE_URL = "https://www.lamoncloa.gob.es"
URL_INDICE = "https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23F.aspx"

HTML_DIR = "./html"
PDF_BASE_DIR = "./pdf"
JSON_OUTPUT_PATH = "./json/listado_de_archivos.json"

# Carga del sitio web principal con Selenium

Descarga del driver más reciente de 'Chrome for Testing':

https://googlechromelabs.github.io/chrome-for-testing/

Inicialización del driver

In [95]:
load_dotenv("../../apis/.env")
chromedriver_path = os.getenv("CHROMEDRIVER_PATH")

service = Service(executable_path=chromedriver_path)
options = webdriver.ChromeOptions()

# Configuramos las preferencias de descarga de Chrome para guardar directamente los PDFs
prefs = {
    "download.default_directory": os.path.abspath(PDF_BASE_DIR),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "plugins.always_open_pdf_externally": True
}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(service=service, options=options)
# driver = uc.Chrome(headless=False) #### USING UNDETECTED CHROMEDRIVER

Carga de la página principal

In [96]:
driver.implicitly_wait(5)
driver.get(URL_INDICE)

time.sleep(0.5)
try:
    accept_cookies_button = driver.find_element(
        By.XPATH, '//*[@id="ctl00_CookieInfo_btnLink"]'
    )
    accept_cookies_button.click()
except:
    pass

## Descarga del contenido del sitio web principal

In [97]:
dd23f_html = driver.page_source
print(dd23f_html[:100],"\n...")
print(dd23f_html[-100:])
# today = pd.Timestamp.today().strftime('%Y_%m_%d')
with open(os.path.join(HTML_DIR, 'dd23f_html_source.html'), 'w', encoding='utf-8') as file:
    file.write(dd23f_html)

<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="es" lang="es" class="no-js"><head id="Head"><me 
...
ookies','event_label': currentDate + ' [' + window.location.pathname + ']'});</script></body></html>


In [98]:
HTML_PATH = os.path.join(HTML_DIR, 'dd23f_html_source.html')
with open(HTML_PATH, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

In [78]:
structure = {}

# Identificamos el <h1> principal 
main_h1 = soup.find("h1")
if not main_h1:
    raise ValueError("No se encontró <h1> en la página.")

# Iteramos sobre cada sección de documentos, identificada por un <h2> dentro de un contenedor con clase "documents-box"
for documents_box in main_h1.find_all_next("div", class_="documents-box"):
    h2 = documents_box.find("h2")
    if not h2:
        continue

    level_1_name = h2.get_text(strip=True)
    structure[level_1_name] = {}

    for li in documents_box.select("ul.docs-list > li"):
        h3 = li.find("h3")
        if not h3:
            continue

        level_2_name = h3.get_text(strip=True)
        structure[level_1_name][level_2_name] = []

        for link in li.find_all("a", href=True):
            href = link["href"]
            if href.lower().endswith(".pdf"):
                full_url = urljoin(BASE_URL, href)
                file_name = link.get_text(strip=True)

                structure[level_1_name][level_2_name].append({
                    "file_name": file_name,
                    "url": full_url
                })

# Guardamos el listado de archivos en un JSON
os.makedirs(PDF_BASE_DIR, exist_ok=True)

with open(JSON_OUTPUT_PATH, "w", encoding="utf-8") as jf:
    json.dump(structure, jf, indent=4, ensure_ascii=False)

print("Listado de archivos guardado en: ", JSON_OUTPUT_PATH)

Listado de archivos guardado en:  ./json/listado_de_archivos.json


## Descarga de los PDFs

In [ ]:
MAX_FILENAME_LENGTH = 50
DOWNLOAD_TIMEOUT = 20

def sanitize_name(name):
    name = re.sub(r'[\\/*?:"<>|]', "", name).strip()
    if len(name) > MAX_FILENAME_LENGTH:
        name = name[:MAX_FILENAME_LENGTH - 3].rstrip() + "..."
    return name


driver.execute_cdp_cmd(
    "Page.setDownloadBehavior",
    {
        "behavior": "allow",
        "downloadPath": os.path.abspath(PDF_BASE_DIR)
    }
)


for level_1, sublevels in structure.items():

    folder_1 = sanitize_name(level_1)

    for level_2, files in sublevels.items():

        folder_2 = sanitize_name(level_2)

        target_folder = os.path.join(PDF_BASE_DIR, folder_1, folder_2)
        os.makedirs(target_folder, exist_ok=True)

        for file_info in files:

            try:
                url = file_info["url"]

                visible_name = sanitize_name(file_info["file_name"]) + ".pdf"
                final_path = os.path.join(target_folder, visible_name)

                # Omitimos descarga si el archivo ya existe
                if os.path.exists(final_path):
                    continue

                print(f"Descargando.... {final_path}")

                filename_from_url = url.split("/")[-1]
                downloaded_path = os.path.join(PDF_BASE_DIR, filename_from_url)

                # Eliminamos cualquier archivo con el mismo nombre que pueda haber quedado de una descarga anterior
                if os.path.exists(downloaded_path):
                    os.remove(downloaded_path)

                driver.get(url)

                start = time.time()

                while time.time() - start < DOWNLOAD_TIMEOUT:
                    if os.path.exists(downloaded_path):
                        break
                    time.sleep(1)

                if os.path.exists(downloaded_path):
                    os.replace(downloaded_path, final_path)

            except Exception:
                continue

Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\RESERVADO parte por abandono de destino del Cap....pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\RESERVADO Hoja de servicios del Cap. Sánchez Va....pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\RESERVADO Informe de Asesoría Jurídica General....pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\SECRETO oficio dando cuenta toma de declaración..pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\SECRETO oficio dando cuenta toma de declaración..pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\RESERVADO oficio dando cuenta toma de declaración..pdf
Descargando.... ./pdf\Ministerio de Defensa\Archivo general e histórico del Ministerio de D...\RESERVADO 

In [90]:
MAX_FILENAME_LENGTH = 50
DOWNLOAD_TIMEOUT = 20

driver.execute_cdp_cmd(
    "Page.setDownloadBehavior",
    {
        "behavior": "allow",
        "downloadPath": os.path.abspath(PDF_BASE_DIR)
    }
)

for level_1, sublevels in structure.items():

    safe_level_1 = re.sub(r'[\\/*?:"<>|]', "", level_1).strip()
    if len(safe_level_1) > MAX_FILENAME_LENGTH:
        safe_level_1 = safe_level_1[:MAX_FILENAME_LENGTH - 3].rstrip() + "..."

    for level_2, files in sublevels.items():

        safe_level_2 = re.sub(r'[\\/*?:"<>|]', "", level_2).strip()
        if len(safe_level_2) > MAX_FILENAME_LENGTH:
            safe_level_2 = safe_level_2[:MAX_FILENAME_LENGTH - 3].rstrip() + "..."

        target_folder = os.path.join(PDF_BASE_DIR, safe_level_1, safe_level_2)
        os.makedirs(target_folder, exist_ok=True)

        for file_info in files:

            url = file_info["url"]
            original_name = file_info["file_name"]

            safe_name = re.sub(r'[\\/*?:"<>|]', "", original_name).strip()

            if len(safe_name) > MAX_FILENAME_LENGTH:
                safe_name = safe_name[:MAX_FILENAME_LENGTH - 3].rstrip() + "..."

            final_path = os.path.join(target_folder, safe_name + ".pdf")

            print(f"Descargando.... {final_path}")

            try:
                before = set(os.listdir(PDF_BASE_DIR))

                driver.get(url)

                start_time = time.time()
                downloaded_file = None

                while time.time() - start_time < DOWNLOAD_TIMEOUT:

                    current = set(os.listdir(PDF_BASE_DIR))
                    new_files = current - before

                    if new_files:

                        candidate = list(new_files)[0]

                        if candidate.endswith(".crdownload"):
                            time.sleep(1)
                            continue
                        else:
                            downloaded_file = candidate
                            break

                    time.sleep(1)

                if not downloaded_file:
                    raise TimeoutError("Download timeout")

                source_path = os.path.join(PDF_BASE_DIR, downloaded_file)

                os.replace(source_path, final_path)

            except Exception:

                error_txt_path = os.path.join(
                    target_folder,
                    safe_name + "_ERROR.txt"
                )

                with open(error_txt_path, "w", encoding="utf-8") as ef:
                    ef.write("ERROR\n")
                    ef.write(f"INTENDED_NAME: {safe_name}.pdf\n")
                    ef.write(f"URL: {url}\n")
                    ef.write(traceback.format_exc())

Descargando.... ./pdf\Ministerio del Interior\Guardia Civil\Transcripción de conversación telefónica de (pr....pdf
Descargando.... ./pdf\Ministerio del Interior\Guardia Civil\Transcripción de conversación telefónica de Gar....pdf
Descargando.... ./pdf\Ministerio del Interior\Guardia Civil\Conversaciones telefónicas de (presuntamente) l....pdf
Descargando.... ./pdf\Ministerio del Interior\Guardia Civil\Documentación con una presunta planificación de....pdf
Descargando.... ./pdf\Ministerio del Interior\Guardia Civil\Documento manuscrito de posible planificación d....pdf
Descargando.... ./pdf\Ministerio del Interior\Guardia Civil\Transcripción de cintas grabadas con conversaci....pdf
Descargando.... ./pdf\Ministerio del Interior\Guardia Civil\Nota del EM de la Guardia Civil con una secuenc....pdf
Descargando.... ./pdf\Ministerio del Interior\Guardia Civil\Télex interiores y de agencias recibidos en 2ª....pdf
Descargando.... ./pdf\Ministerio del Interior\Guardia Civil\Oficio zona País Vasc

FileNotFoundError: [Errno 2] No such file or directory: './pdf\\Ministerio de Defensa\\Archivo general e histórico del Ministerio de D...\\RESERVADO parte por abandono de destino del Cap..._ERROR.txt'

______________

## Prueba de descarga con URL específica

In [ ]:
# PRUEBA

test_pdf_url = "https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civil/23F_3._Conversaciones_telefonicas_unidad_militar_El_Pardo.pdf"

os.makedirs(PDF_BASE_DIR, exist_ok=True)

# Force Chrome download behavior (important)
driver.execute_cdp_cmd(
    "Page.setDownloadBehavior",
    {
        "behavior": "allow",
        "downloadPath": os.path.abspath(PDF_BASE_DIR)
    }
)

print("Descargando PDF desde:", test_pdf_url)

before_files = set(os.listdir(PDF_BASE_DIR))

driver.get(test_pdf_url)

while True:
    current_files = set(os.listdir(PDF_BASE_DIR))
    new_files = current_files - before_files

    if new_files:
        if any(f.endswith(".crdownload") for f in new_files):
            time.sleep(1)
            continue
        else:
            break
    time.sleep(1)

downloaded_file = list(new_files)[0]

print("Guardado PDF en:", os.path.abspath(os.path.join(PDF_BASE_DIR, downloaded_file)))

Starting single PDF download test...
Download completed.
File saved as: 23F_3._Conversaciones_telefonicas_unidad_militar_El_Pardo.pdf
Location: c:\Users\david\Documents\git\documentos-desclasificados-23f\pdf\23F_3._Conversaciones_telefonicas_unidad_militar_El_Pardo.pdf


In [56]:
driver.close()